# Pipeline — Negociação Secundária de Crédito Privado

Cada bloco é um fluxo — rode isoladamente o que precisar. A lógica está em `scripts/pipeline_core.py`: cada fluxo é uma função (`pc.boletim`, `pc.ntnb`, `pc.calc_taxa`, ...) que chama o CLI do script correspondente. **O CLI vive só no `pipeline_core`** — se um script mudar de argumentos, altera-se lá e todos os blocos/rotinas se ajustam. Nada aborta o notebook: cada passo mostra `[OK]`/`[FALHA]`.

**Rode este notebook a partir da pasta `code/`.** Ordem e parâmetros: ver `vault/11 - Pipeline de Execucao.md` e `vault/13 - Migracao Banco.md`.

Seções: **A)** testar cada fluxo isolado · **B)** rotina diária · **C)** setup em massa.

## Setup do notebook (rodar primeiro)

In [ ]:
import sys
from pathlib import Path

scripts = Path.cwd() / "scripts"
if not scripts.exists():
    raise SystemExit(f"Rode o notebook a partir da pasta code/. cwd atual: {Path.cwd()}")
sys.path.insert(0, str(scripts))
import pipeline_core as pc

# Datas de teste (dia util mais recente e o anterior). Ajuste se quiser.
X    = pc.ultimos_n_dias_uteis(1)[0].isoformat()
Xant = pc.dia_util_anterior(X).isoformat()
print("Data de teste X =", X, "| X-1u =", Xant)

# A) Testar cada fluxo isolado
Rode um bloco de cada vez e confira o `[OK]`/`[FALHA]`. No banco o ambiente muda (proxy, Playwright, login, Bloomberg) — **o que quebrar, me avise.** Cada bloco chama uma função de fluxo do `pipeline_core`.

### Scraping (rede)

In [ ]:
# 1. Boletim B3 (negocios) — precisa de X-1u e X
pc.boletim(Xant, X)

In [ ]:
# 2. Anbima debentures (taxa indicativa)
pc.anbima_deb(X)

In [ ]:
# 3. Anbima CRI/CRA (taxa indicativa) — Playwright
pc.anbima_cricra(X)

In [ ]:
# 4. FI Analytics planilha (caracteristicas) — Playwright + login
pc.fianalytics()

In [ ]:
# 5. Anbima Data (caracteristicas + fluxo) — Playwright
pc.anbima_data(Xant, X)

In [ ]:
# 6. Anbima NTN-B (MtM) — precisa de X-1u e X
pc.ntnb(Xant, X)

In [ ]:
# 7. Curva DI B3 (MtM) — rodar para X-1u e X
pc.curva_di(Xant)
pc.curva_di(X)

In [ ]:
# 8. Outstanding via Bloomberg — SO NO BANCO (terminal Bloomberg logado)
pc.outstanding(X)

### Cálculo (local — sem rede)

In [ ]:
# 9. Calcular taxa por trade (cascata FI Analytics -> B3) — ANTES de filtrar
pc.calc_taxa(X)

In [ ]:
# 10. Filtrar (VALIDO / FUNDO / BROKER / PF)
pc.filtrar(X)

In [ ]:
# 11. Spread Anbima das indicativas
pc.spread_anbima(X)

In [ ]:
# 12. Match de referencia (global, sem data) — ANTES de spread_over
pc.match_ref()

In [ ]:
# 13. Spread over dos trades
pc.spread_over(X)

In [ ]:
# 14. Gerar relatorio (toda a base)
pc.relatorio()

# B) Rotina diária
Roda a cadeia completa para os últimos `N_DIAS` dias úteis (pega correções retroativas) e gera o relatório no fim. Ajuste `N_DIAS`.

In [ ]:
N_DIAS = 5   # quantos dias uteis reprocessar
pc.run_ultimos_n(N_DIAS)

Rodar um único dia de liquidação inteiro:

In [ ]:
DIA = X   # ajuste a data de liquidacao
pc.run_dia(DIA)

# C) Setup inicial (bulk — rodar 1 vez, depois de validar a Seção A)
Bootstrap da base: raspa o histórico largo de cada fonte (Anbima Data completa; deb/NTN-B ~4 meses; curva DI ~20 pregões; CRI/CRA ~5 pregões; boletim desde `INICIO_BOLETIM`) e roda a cadeia de cálculo. **Tolerante a falha:** segue em frente e traz o que cada fonte tiver, com resumo `[OK]`/`[FALHA]` no fim. **Demorado.**

In [ ]:
INICIO_BOLETIM = "2026-03-02"   # ~4 meses atras; define o historico de negocios do relatorio
OUTSTANDING    = False          # True apenas no banco (terminal Bloomberg)

pc.run_setup(INICIO_BOLETIM, rodar_outstanding=OUTSTANDING)